# Extract training samples
* Download low-tide cloud free satellite iamges closest to the UAV image collection
* Sample the satellite image bands where appromximately a single UAV class

In [1]:
import pathlib
import numpy
import dask.distributed

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

C:\Users\pearsonra\.conda\envs\satellite\Lib\site-packages\dask\array\chunk_types.py:131: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  import scipy.sparse
C:\Users\pearsonra\.conda\envs\satellite\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


# Values to edit

In [2]:
sample_method = "sampling_2" # sampling_1 sampling_2
method_2_threshold = .8 # .95 .97 .98 .99 1.0
max_cloud_cover = 5 # percentage
low_tide_delta_hrs = 0
low_tide_delta_mins = 30

site_names = ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", "Ihutai",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

# Cells to run

In [3]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:53558,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 8
Started: Just now,Total memory: 31.73 GiB
Comm: tcp://127.0.0.1:53583,Total threads: 2
Dashboard: http://127.0.0.1:53584/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:53561,


In [4]:
data_path = utils.get_data_path()
utils.create_data_folders()

training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
lowtide_search_range_file = data_path / "ELF24505_satellite_day_search_range.csv"
servey_dates_file = data_path / "ELF24505_SurveyDates.csv" 
uav_folder = data_path / "classified_uav"
samples_folder = utils.get_samples_path(sample_method=sample_method, method_2_threshold=method_2_threshold,
                                        max_cloud_cover=max_cloud_cover, low_tide_delta_hrs=low_tide_delta_hrs,
                                        low_tide_delta_mins=low_tide_delta_mins)

In [7]:
for site_name in site_names:
    sampling.sample_site(
        site_name=site_name,
        training_labels_file=training_labels_file,
        uav_folder=uav_folder,
        sample_method=sample_method,
        method_2_threshold=method_2_threshold,
        max_cloud_cover=max_cloud_cover,
        low_tide_delta_hrs=low_tide_delta_hrs,
        low_tide_delta_mins=low_tide_delta_mins
    )
counts_summary = sampling.site_sample_counts_by_class(sample_method=sample_method, method_2_threshold=method_2_threshold,
                                                      max_cloud_cover=max_cloud_cover, low_tide_delta_hrs=low_tide_delta_hrs,
                                                      low_tide_delta_mins=low_tide_delta_mins)

Site CatlinsLake
Site CatlinsRiverMouth
Site Childrens
Site Duvauchelle
Site Robinsons
Site Takamatua
Site Purau
Site Ihutai
Site IveyBay_Nov25
Site IveyBay_Feb26
Site LeftBank_Nov25
Site LeftBank_Feb26
Site Paremata_Nov25
Site Paremata_Feb26
Site Paremata_Feb25
	WARNING - satellite image for site Paremata_Feb25 - either it's not yet downloaded or	 there is no suitable low cloud / low tide satellite image. Ignoring this site.	Try running `get_site_satellite` first if you expect a satellite image.
Site ThePoint_Nov25
Construct training data from UAV and Satellite imagery
	Sample satellite 1 of 1
		Class Seagrass - 0 samples
		Class Seagrass submerged - 0 samples
		Class Gracilaria - 0 samples
		Class Gracilaria submerged - 0 samples
		Class Ulva - 44 samples
		Class Cystophora - 0 samples
		Class Hormosira - 0 samples
		Class Brown algae mixed - 0 samples
		Class Submerged vegetation - 0 samples
		Class Microphytobenthos - 0 samples
		Class Green algae mixed - 0 samples
		Class Filament

In [8]:
counts_summary[['Seagrass', 'Seagrass submerged', 'Gracilaria', 'Gracilaria submerged', 'Ulva', 'Ulva mats', 'Unvegetated',
'Water', 'Terrestrial', 'Submerged vegetation', 'Microphytobenthos', 'Rock', 'Saltmarsh' , 'Shadow', 'Glare']].astype(int) 

uav_class_name,Seagrass,Seagrass submerged,Gracilaria,Gracilaria submerged,Ulva,Ulva mats,Unvegetated,Water,Terrestrial,Submerged vegetation,Microphytobenthos,Rock,Saltmarsh,Shadow,Glare
Site,,,,,,,,,,,,,,,
CatlinsLake,0,0,1532,1533,0,0,25908,7112,0,0,0,0,0,0,0
CatlinsRiverMouth,374,0,0,0,0,0,1979,179,829,0,0,90,0,0,42
Childrens,127,6,0,0,0,0,492,723,0,4,18,0,0,0,0
Duvauchelle,1120,280,0,0,0,0,1072,2718,0,0,30,0,0,7,0
Ihutai,6711,0,7209,0,1263,5070,105399,24861,1074,10407,0,0,4719,0,0
IveyBay_Feb26,17,0,0,0,0,0,46,0,1,0,0,0,0,0,0
IveyBay_Nov25,3,0,0,0,0,0,79,3,1,0,0,0,0,0,0
IveyBay_ThePoint_LeftBank_Oct24,5,0,0,0,0,0,94,205,0,23,0,0,0,6,0
LeftBank_Feb26,54,0,0,0,0,0,175,92,0,0,0,0,0,0,0


In [9]:
print(f"Samples located at {samples_folder}")

Samples located at C:\Local\repos\seagrass-detection\data\training\low_tide_delta_30mins_max_cloud_percentage_5\sampling_2_80_percent
